In [4]:
!pip install torch tensorflow>=2.12 tqdm>=4.66 gpt_download3

ERROR: Could not find a version that satisfies the requirement gpt_download3 (from versions: none)
ERROR: No matching distribution found for gpt_download3


In [2]:
import torch as th
import tensorflow as tf
import tqdm

In [121]:
from transformers import GPT2LMHeadModel

# 1. Download the official OpenAI GPT-2 124M weights
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_model.eval()
# 2. Extract the state dictionary containing all weight tensors
params = hf_model.state_dict()

# 3. View the available parameter names (keys)
for key in list(params.keys())[:10]:
    print(f"Key: {key:<40} Shape: {params[key].shape}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Key: transformer.wte.weight                   Shape: torch.Size([50257, 768])
Key: transformer.wpe.weight                   Shape: torch.Size([1024, 768])
Key: transformer.h.0.ln_1.weight              Shape: torch.Size([768])
Key: transformer.h.0.ln_1.bias                Shape: torch.Size([768])
Key: transformer.h.0.attn.c_attn.weight       Shape: torch.Size([768, 2304])
Key: transformer.h.0.attn.c_attn.bias         Shape: torch.Size([2304])
Key: transformer.h.0.attn.c_proj.weight       Shape: torch.Size([768, 768])
Key: transformer.h.0.attn.c_proj.bias         Shape: torch.Size([768])
Key: transformer.h.0.ln_2.weight              Shape: torch.Size([768])
Key: transformer.h.0.ln_2.bias                Shape: torch.Size([768])


In [122]:
def load_weights(model, params):

    # Token and positional embeddings
    model.token_embed.weight.data = params["transformer.wte.weight"]
    model.positional_embed.weight.data = params["transformer.wpe.weight"]

    # Transformer blocks
    for b_idx, block in enumerate(model.transformer_block):

        prefix = f"transformer.h.{b_idx}"

        # -------------------------
        # LayerNorm 1
        # -------------------------
        block.norm1.scale.data = params[f"{prefix}.ln_1.weight"]
        block.norm1.shift.data = params[f"{prefix}.ln_1.bias"]

        # -------------------------
        # QKV
        # -------------------------
        qkv_weight = params[f"{prefix}.attn.c_attn.weight"]
        qkv_bias = params[f"{prefix}.attn.c_attn.bias"]

        q_w, k_w, v_w = qkv_weight.split(768, dim=-1)
        q_b, k_b, v_b = qkv_bias.split(768, dim=0)

        block.mha.wq.weight.data = q_w.T
        block.mha.wk.weight.data = k_w.T
        block.mha.wv.weight.data = v_w.T

        # Assign biases since qkv_bias is now True
        block.mha.wq.bias.data = q_b
        block.mha.wk.bias.data = k_b
        block.mha.wv.bias.data = v_b

        # -------------------------
        # Attention output projection
        # -------------------------
        block.mha.proj.weight.data = (
            params[f"{prefix}.attn.c_proj.weight"].T
        )

        block.mha.proj.bias.data = (
            params[f"{prefix}.attn.c_proj.bias"]
        )

        # -------------------------
        # LayerNorm 2
        # -------------------------
        block.norm2.scale.data = params[f"{prefix}.ln_2.weight"]
        block.norm2.shift.data = params[f"{prefix}.ln_2.bias"]

        # -------------------------
        # Feed Forward
        # -------------------------
        block.ffnn.layers[0].weight.data = (
            params[f"{prefix}.mlp.c_fc.weight"].T
        )

        block.ffnn.layers[0].bias.data = (
            params[f"{prefix}.mlp.c_fc.bias"]
        )

        block.ffnn.layers[2].weight.data = (
            params[f"{prefix}.mlp.c_proj.weight"].T
        )

        block.ffnn.layers[2].bias.data = (
            params[f"{prefix}.mlp.c_proj.bias"]
        )

    # Final LayerNorm
    model.layer_norm.scale.data = params["transformer.ln_f.weight"]
    model.layer_norm.shift.data = params["transformer.ln_f.bias"]

    # GPT-2 ties output weights to token embeddings
    model.output.weight.data = model.token_embed.weight.data

    model.eval()

In [123]:

from torch import nn

class GPTModel(th.nn.Module):
    def __init__ (self,config):
        super().__init__()

        self.config=config

        self.token_embed=th.nn.Embedding(config["vocab_size"],config["embed_dim"])
        self.positional_embed=th.nn.Embedding(config["context_length"],config["embed_dim"])
        self.dropout=th.nn.Dropout(config["dropout_rate"])

        self.transformer_block=th.nn.Sequential(*[Transformer(config) for _ in range(config["num_layers"])])
        self.layer_norm=LayerNorm(config["embed_dim"])
        self.output=th.nn.Linear(config["embed_dim"],config["vocab_size"],bias=False)

    def forward (self,x):
        # x is expected to be (batch_size, sequence_length) containing token IDs
        token_embed=self.token_embed(x)
        positional_embed=self.positional_embed(th.arange(x.shape[1],device=x.device))
        x=token_embed+positional_embed
        x=self.dropout(x)
        x=self.transformer_block(x)
        x=self.layer_norm(x)
        logits=self.output(x)
        return logits

class MultiHeadAttention(th.nn.Module):
    def __init__(self,d_in,d_out,dropout,num_heads,context_length,qkv_bias=False):
        super().__init__()
        assert (d_out% num_heads==0), "d_out must be divisible by num_heads"
        self.dropout=nn.Dropout(dropout)
        self.d_out=d_out
        self.wk=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.wq=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.wv=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.num_heads=num_heads
        self.proj=nn.Linear(d_out,d_out)

        self.register_buffer("mask",th.triu(th.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b,num_tokens,d_in=x.shape

        keys=self.wk(x)
        queries=self.wq(x)
        values=self.wv(x)

        keys=keys.view(b,num_tokens,self.num_heads,self.d_out//self.num_heads)
        queries=queries.view(b,num_tokens,self.num_heads,self.d_out//self.num_heads)
        values=values.view(b,num_tokens,self.num_heads,self.d_out//self.num_heads)

        keys=keys.transpose(-3,-2)
        queries=queries.transpose(-3,-2)
        values=values.transpose(-3,-2)

        attention_scores=queries @ keys.transpose(2,3)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-th.inf)

        attention_weights=th.softmax(attention_scores/keys.shape[-1]**0.5,dim=-1)

        norm=self.dropout(attention_weights)

        context_vector =(norm @ values).transpose(1,2)

        context_vector=context_vector.contiguous().view(b,num_tokens,self.d_out)

        context_vector = self.proj(context_vector)


        context_vector_proj=self.dropout(context_vector)

        return context_vector_proj


class Transformer(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.mha=MultiHeadAttention(
            d_in=config["embed_dim"],
            d_out=config["embed_dim"],
            num_heads=config["num_heads"],
            dropout=config["dropout_rate"],
            context_length=config["context_length"],
            qkv_bias=config["qkv_bias"]
        )
        self.ffnn=FeedForwardLayer(config)
        self.norm1=LayerNorm(embed_dim=config["embed_dim"],epsilon=1e-5)
        self.norm2=LayerNorm(embed_dim=config["embed_dim"],epsilon=1e-5)
        self.dropout_rate=nn.Dropout(config["dropout_rate"])

    def forward(self,x):

        input=x

        x=self.norm1(x)
        x=self.mha(x)
        x=self.dropout_rate(x)
        x=x+input



        input=x

        x=self.norm2(x)
        x=self.ffnn(x)
        x=self.dropout_rate(x)
        x=x+input

        return x

class GeLU(th.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (
            1 + th.tanh(
                (2 / th.pi) ** 0.5 *
                (x + 0.044715 * x**3)
            )
        )

class FeedForwardLayer(th.nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config # Store config as an instance variable
        self.layers=th.nn.Sequential(
                th.nn.Linear(self.config["embed_dim"],4*self.config["embed_dim"]),
                GeLU(),
                th.nn.Linear(4*self.config["embed_dim"],self.config["embed_dim"])
        )

    def forward(self,x):
        return self.layers(x)



class LayerNorm(th.nn.Module):
    def __init__ (self,embed_dim,epsilon=1e-5):
        super().__init__()
        self.epsilon=epsilon
        self.scale=th.nn.Parameter(th.ones(embed_dim))
        self.shift=th.nn.Parameter(th.zeros(embed_dim))

    def forward (self,x):
        x_mean=x.mean(dim=-1,keepdim=True)
        x_variance=x.var(dim=-1,keepdim=True,unbiased=False)
        x_norm=(x-x_mean)/th.sqrt(x_variance+self.epsilon)

        return self.scale*x_norm + self.shift

In [124]:
GPT_CONFIG_124M = {
    "vocab_size":50257,
    "context_length":1024,
    "embed_dim":768,
    "num_heads":12,
    "num_layers":12,
    "dropout_rate":0.1,
    "qkv_bias":True # Change to True to include biases
}

In [125]:
model=GPTModel(GPT_CONFIG_124M)
load_weights(model,params)

In [152]:
model.transformer_block[0].mha.proj

Linear(in_features=768, out_features=768, bias=True)

In [126]:
model

GPTModel(
  (token_embed): Embedding(50257, 768)
  (positional_embed): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_block): Sequential(
    (0): Transformer(
      (mha): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace=False)
        (wk): Linear(in_features=768, out_features=768, bias=True)
        (wq): Linear(in_features=768, out_features=768, bias=True)
        (wv): Linear(in_features=768, out_features=768, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ffnn): FeedForwardLayer(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GeLU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout_rate): Dropout(p=0.1, inplace=False)
    )
    (1): Transformer(
      (mha): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace

In [127]:
def generate_text(gpt,idx,context_size,new_max_tokens):


    for _ in range(new_max_tokens):
        # Ensure the input to the model does not exceed context_size
        # Take the last `context_size` tokens, or fewer if the sequence is shorter.
        input_to_gpt = idx[:, - context_size:]

        with th.no_grad():
            # Pass the context-limited input to the model
            logits=gpt(input_to_gpt)

        # Get the logits for the last token in the processed sequence
        logits=logits[:,-1,:]

        top_k_logits,top_k_tokens=th.topk(logits,4)

        min_prob=top_k_logits.min()
        logits=th.where(condition=logits<min_prob,other=logits,input=th.tensor(-th.inf).to(logits.device))

        scaled_logits=logits/1.3
        probs=th.nn.functional.softmax(scaled_logits,dim=-1)
        nxt_idx=th.multinomial(probs,num_samples=1)


        idx=th.cat((idx,nxt_idx),dim=1)

    return idx

In [182]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")


In [201]:
input="My name is Anbarasan and I'm an AI research scientist"

In [202]:
input_ids = tokenizer.encode(input)

print(input_ids)
decoded_ids=generate_text(model,th.tensor(input_ids).unsqueeze(0),context_size=GPT_CONFIG_124M["context_length"],new_max_tokens=16)

[3666, 1438, 318, 1052, 5657, 292, 272, 290, 314, 1101, 281, 9552, 2267, 11444]


In [203]:
tokenizer.decode(decoded_ids[0].tolist())

"My name is Anbarasan and I'm an AI research scientist and the creator and creator behind The AI. I'm a big believer and believer"

In [163]:
model.eval()

with th.no_grad():
    model_logits = model(th.tensor(input_ids).unsqueeze(0))

In [164]:
hf_model.eval()

with th.no_grad():
    hf_logits = hf_model(th.tensor(input_ids).unsqueeze(0))

In [165]:
hf_logits

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -33.0735,  -32.3348,  -35.2379,  ...,  -38.3576,  -38.4758,
           -33.0943],
         [ -56.9638,  -55.8813,  -63.3229,  ...,  -66.8965,  -69.3888,
           -61.2562],
         [ -70.2709,  -69.6677,  -74.4322,  ...,  -78.1296,  -77.4711,
           -71.8745],
         ...,
         [ -93.2652,  -94.4709,  -99.9113,  ..., -106.0784, -101.9572,
           -97.3128],
         [ -81.9240,  -82.0695,  -84.3274,  ...,  -87.1274,  -88.7986,
           -82.8689],
         [ -73.0115,  -72.1000,  -78.2209,  ...,  -82.6835,  -82.1261,
           -73.9849]]]), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [166]:
print("My model:", model_logits.shape)
print("HF model:", hf_logits["logits"].shape)

print(
    th.max(
        th.abs(model_logits - hf_logits["logits"])
    )
)

My model: torch.Size([1, 14, 50257])
HF model: torch.Size([1, 14, 50257])
tensor(7.6294e-05)


In [116]:
text = "Machine Learning"

input_ids = tokenizer.encode(text)
idx = th.tensor(input_ids).unsqueeze(0)

model.eval()
hf_model.eval()

with th.no_grad():

    # Embedding
    my_x = model.token_embed(idx)
    hf_x = hf_model.transformer.wte(idx)

    print(
        "Embedding:",
        th.max(th.abs(my_x - hf_x)).item()
    )

    # Position embedding
    positions = th.arange(
        idx.shape[1],
        device=idx.device
    )

    my_x = my_x + model.positional_embed(positions)

    hf_x = hf_x + hf_model.transformer.wpe(positions)

    print(
        "Embedding + position:",
        th.max(th.abs(my_x - hf_x)).item()
    )

Embedding: 0.0
Embedding + position: 0.0


In [117]:
with th.no_grad():
    my_norm = model.transformer_block[0].norm1(my_x)
    hf_norm = hf_model.transformer.h[0].ln_1(hf_x)

print(
    "LayerNorm:",
    th.max(th.abs(my_norm - hf_norm)).item()
)

LayerNorm: 5.960464477539063e-08


In [118]:
with th.no_grad():

    my_q = model.transformer_block[0].mha.wq(my_norm)
    my_k = model.transformer_block[0].mha.wk(my_norm)
    my_v = model.transformer_block[0].mha.wv(my_norm)

    hf_qkv = hf_model.transformer.h[0].attn.c_attn(hf_norm)

    hf_q, hf_k, hf_v = hf_qkv.split(768, dim=-1)

    print("Q:", th.max(th.abs(my_q - hf_q)).item())
    print("K:", th.max(th.abs(my_k - hf_k)).item())
    print("V:", th.max(th.abs(my_v - hf_v)).item())

Q: 1.230086326599121
K: 1.3112515211105347
V: 0.35575786232948303


In [119]:
hf_qkv_weight = params["transformer.h.0.attn.c_attn.weight"]

q_w, k_w, v_w = hf_qkv_weight.split(768, dim=-1)

print("HF q_w:", q_w.shape)
print("My q_w:", model.transformer_block[0].mha.wq.weight.shape)

print(
    "Q weight difference:",
    th.max(
        th.abs(
            model.transformer_block[0].mha.wq.weight - q_w.T
        )
    ).item()
)

HF q_w: torch.Size([768, 768])
My q_w: torch.Size([768, 768])
Q weight difference: 0.0


In [120]:
hf_qkv_bias = params["transformer.h.0.attn.c_attn.bias"]

q_b, k_b, v_b = hf_qkv_bias.split(768, dim=0)

print(
    "Q bias difference:",
    th.max(
        th.abs(
            model.transformer_block[0].mha.wq.bias - q_b
        )
    ).item()
)

Q bias difference: 1.2300862073898315
